# Week 4 - Task 7: ML Pipeline with Feature Engineering

This task builds a professional Machine Learning pipeline using Scikit-learn. 
The pipeline combines feature engineering, preprocessing, and model training into 
a single reusable workflow. The Titanic dataset is used to compare a pipeline 
with the original features against a pipeline with engineered features.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

## Loading the Dataset

The Titanic dataset is loaded from the `train.csv` file using Pandas.
After loading the dataset, we display the first few rows to verify that
the data has been imported correctly.

In [3]:
df = pd.read_csv("train.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


##  Understanding the Dataset

Before building the Machine Learning pipeline, we inspect the dataset to understand
its structure, columns, data types, and missing values.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


##  Checking Missing Values

Missing values can affect model training. We check the number of missing values
in each column before defining our preprocessing pipeline.

In [5]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

##  Separating Features and Target

The `Survived` column is our target variable because the goal is to predict
whether a passenger survived or not.

We remove `Survived` from the input features and also remove `PassengerId`,
`Name`, `Ticket`, and `Cabin` because these columns are not suitable for our
initial classification pipeline.

In [6]:
X = df.drop(columns=["Survived", "PassengerId", "Name", "Ticket", "Cabin"])
y = df["Survived"]

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


##  Train-Test Split

The dataset is divided into training and testing sets. The training data is used
to learn the preprocessing parameters and model, while the test data is kept
separate for evaluating the final performance.

A fixed `random_state` ensures that the split is reproducible.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 712
Testing samples: 179


##  Identifying Numerical and Categorical Features

The features are divided into numerical and categorical columns.

Numerical features will be standardized using `StandardScaler`, while categorical
features will be converted into numerical form using `OneHotEncoder`.

This separation allows the `ColumnTransformer` to apply the appropriate
preprocessing to each type of feature.

In [8]:
numerical_features = ["Age", "SibSp", "Parch", "Fare"]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['Age', 'SibSp', 'Parch', 'Fare']
Categorical features: ['Pclass', 'Sex', 'Embarked']


##  Building the Preprocessing Pipeline

A `ColumnTransformer` is used to apply different preprocessing techniques
to numerical and categorical features.

For numerical features:
- Missing values are filled using the median.
- `StandardScaler` standardizes the numerical values.

For categorical features:
- Missing values are filled using the most frequent value.
- `OneHotEncoder` converts categorical values into numerical features.

This preprocessing will later be combined with the Logistic Regression model
inside a single Machine Learning pipeline.

In [9]:
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


##  Building the Complete Machine Learning Pipeline

The preprocessing step and Logistic Regression model are combined into one
Pipeline object.

This ensures that the same preprocessing steps are consistently applied to
both training and testing data and helps prevent data leakage.

The complete pipeline can also be saved and reused later without manually
repeating the preprocessing steps.

In [10]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

print("Complete ML pipeline created successfully!")

Complete ML pipeline created successfully!


##  Training the Machine Learning Pipeline

The complete pipeline is fitted on the training data.

Because preprocessing and model training are combined inside the pipeline,
the preprocessing steps are learned only from the training data.

In [11]:
model_pipeline.fit(X_train, y_train)

print("Pipeline trained successfully!")

Pipeline trained successfully!


##  Evaluating the Pipeline

The trained pipeline is used to make predictions on the unseen test data.
Accuracy, classification report, and confusion matrix are used to evaluate
the model's performance.

In [12]:
y_pred = model_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Pipeline Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Pipeline Accuracy: 0.8044692737430168

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       110
           1       0.79      0.67      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

Confusion Matrix:
[[98 12]
 [23 46]]


##  Feature Engineering

Feature engineering involves creating new meaningful features from existing
data to help the Machine Learning model identify useful patterns.

For the Titanic dataset, we create two new features:

- `FamilySize`: Total number of people in the passenger's family group,
  including the passenger.
- `IsAlone`: Indicates whether the passenger was traveling alone.

In [13]:
def create_features(data):
    data = data.copy()

    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)

    return data


X_engineered = create_features(X)

X_engineered.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone
0,3,male,22.0,1,0,7.2500,S,2,0
1,1,female,38.0,1,0,71.2833,C,2,0
2,3,female,26.0,0,0,7.9250,S,1,1
3,1,female,35.0,1,0,53.1000,S,2,0
4,3,male,35.0,0,0,8.0500,S,1,1


##  Train-Test Split with Engineered Features

The newly engineered features are included in the dataset before splitting it
into training and testing sets. The same 80/20 split and random state are used
to make the comparison with the baseline pipeline fair.

In [14]:
X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(
    X_engineered,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train_eng.shape[0])
print("Testing samples:", X_test_eng.shape[0])

Training samples: 712
Testing samples: 179


##  Updating Feature Lists

The two engineered features, `FamilySize` and `IsAlone`, are added to the
numerical features because they contain numerical values.

In [15]:
numerical_features_eng = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]

categorical_features_eng = [
    "Pclass",
    "Sex",
    "Embarked"
]

print("Updated numerical features:", numerical_features_eng)
print("Categorical features:", categorical_features_eng)

Updated numerical features: ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
Categorical features: ['Pclass', 'Sex', 'Embarked']


##  Building the Preprocessing Pipeline with Engineered Features

The preprocessing pipeline is updated to include the newly created
`FamilySize` and `IsAlone` features.

Numerical features are imputed and standardized, while categorical features
are imputed and one-hot encoded using the same preprocessing approach.

In [16]:
numerical_transformer_eng = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_eng = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_eng = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer_eng, numerical_features_eng),
        ("cat", categorical_transformer_eng, categorical_features_eng)
    ]
)

print("Engineered preprocessing pipeline created successfully!")

Engineered preprocessing pipeline created successfully!


##  Building the Engineered Machine Learning Pipeline

The engineered preprocessing step is combined with Logistic Regression into
one complete Pipeline object.

This allows feature preprocessing and model training to be performed
consistently as a single workflow.

In [17]:
engineered_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_eng),
    ("model", LogisticRegression(max_iter=1000))
])

print("Engineered ML pipeline created successfully!")

Engineered ML pipeline created successfully!


##  Training the Engineered Pipeline

The engineered pipeline is trained using the training data containing the
newly created features.

In [18]:
engineered_pipeline.fit(X_train_eng, y_train_eng)

print("Engineered pipeline trained successfully!")

Engineered pipeline trained successfully!


##  Evaluating the Engineered Pipeline

The engineered pipeline is evaluated on the unseen test data. Its accuracy
will be compared with the baseline pipeline to determine whether the new
features improved model performance.

In [19]:
y_pred_eng = engineered_pipeline.predict(X_test_eng)

engineered_accuracy = accuracy_score(y_test_eng, y_pred_eng)

print("Engineered Pipeline Accuracy:", engineered_accuracy)

print("\nClassification Report:")
print(classification_report(y_test_eng, y_pred_eng))

print("Confusion Matrix:")
print(confusion_matrix(y_test_eng, y_pred_eng))

Engineered Pipeline Accuracy: 0.8156424581005587

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       110
           1       0.81      0.68      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179

Confusion Matrix:
[[99 11]
 [22 47]]


##  Comparing Baseline and Engineered Pipelines

The baseline pipeline and the engineered pipeline are compared using their
test accuracy.

The baseline pipeline uses the original features, while the engineered
pipeline additionally uses `FamilySize` and `IsAlone`.

In [20]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline Pipeline",
        "Engineered Pipeline"
    ],
    "Accuracy": [
        accuracy,
        engineered_accuracy
    ]
})

comparison["Accuracy (%)"] = comparison["Accuracy"] * 100

comparison

,Model,Accuracy,Accuracy (%)
0,Baseline Pipeline,0.804469,80.446927
1,Engineered Pipeline,0.815642,81.564246


##  Feature Engineering Result

The engineered pipeline achieved higher test accuracy than the baseline
pipeline. The addition of `FamilySize` and `IsAlone` improved the accuracy
from approximately 80.45% to 81.56%.

This shows that meaningful feature engineering can help the model learn
additional patterns from the existing data.

In [21]:
improvement = engineered_accuracy - accuracy

print(f"Baseline Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Engineered Accuracy: {engineered_accuracy:.4f} ({engineered_accuracy * 100:.2f}%)")
print(f"Improvement: {improvement:.4f} ({improvement * 100:.2f} percentage points)")

Baseline Accuracy: 0.8045 (80.45%)
Engineered Accuracy: 0.8156 (81.56%)
Improvement: 0.0112 (1.12 percentage points)


##  Saving the Final Machine Learning Pipeline

The engineered pipeline is saved using `joblib`.

Saving the complete pipeline allows the preprocessing steps and trained model
to be reused later without rebuilding the entire workflow.

In [22]:
joblib.dump(
    engineered_pipeline,
    "titanic_ml_pipeline.joblib"
)

print("Final pipeline saved successfully!")

Final pipeline saved successfully!


##  Loading the Saved Pipeline

The saved pipeline is loaded again to verify that it can be reused for
predictions after being saved.

In [23]:
loaded_pipeline = joblib.load("titanic_ml_pipeline.joblib")

loaded_predictions = loaded_pipeline.predict(X_test_eng)

loaded_accuracy = accuracy_score(y_test_eng, loaded_predictions)

print(f"Loaded Pipeline Accuracy: {loaded_accuracy:.4f}")

Loaded Pipeline Accuracy: 0.8156


##  Conclusion

A complete Machine Learning pipeline was successfully developed using
Scikit-learn's `Pipeline` and `ColumnTransformer`.

The baseline pipeline achieved an accuracy of **80.45%**, while the pipeline
with the engineered `FamilySize` and `IsAlone` features achieved **81.56%**.
This represents an improvement of approximately **1.12 percentage points**.

The final engineered pipeline was saved using `joblib` and successfully
loaded again, confirming that it can be reused for future predictions.

This pipeline approach makes the workflow cleaner, reusable, consistent, and
less prone to preprocessing errors and data leakage.

In [24]:
import os

print("Files in Task7 folder:")
for file in os.listdir():
    print("-", file)

Files in Task7 folder:
- ML_Pipeline_Feature_Engineering.ipynb
- titanic_ml_pipeline.joblib
- train.csv
